[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/02_erstes_llm.ipynb)

# Sitzung 2 — Das Produkt & ein LLM zum Laufen bringen

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI**

Heute schreibt und startet ihr zum ersten Mal selbst Code: ein LLM ordnet eine Produktbewertung ein. Kein Vorwissen nötig — wir gehen Zelle für Zelle.

> 💡 Ihr müsst nichts installieren und keinen Account anlegen. Alles läuft hier im Browser. Für heute nutzen wir einen **eingebauten „Mock“** (eine Attrappe), damit alle sofort loslegen können — ohne Schlüssel.

## 0. Setup — einmal ausführen

Führt die nächste Zelle aus (Klick hinein, dann **Umschalt+Enter**). Sie lädt eine Bibliothek, die wir *später* für den echten LLM-Aufruf brauchen. Für den Mock heute ist sie nicht nötig — wir machen es trotzdem gleich richtig.

> ⚠️ **Wichtig:** In Colab ist nach jedem Neustart alles weg. Wenn ihr das Notebook neu öffnet, diese Zelle **erneut** ausführen.

In [ ]:
# Einmal pro Sitzung ausführen:
!pip install anthropic --quiet
print('Fertig. Bibliothek geladen.')

## 1. Was ist ein „LLM-Aufruf“?

Ganz einfach: **Text rein → Text raus.** Wir schicken einem Sprachmodell eine Bewertung und eine Anweisung („ordne das Sentiment ein“) und bekommen ein Ergebnis zurück.

Die nächste Zelle definiert unser Werkzeug. Ihr müsst den Code nicht im Detail verstehen — lest die Kommentare, führt sie aus. Wichtig ist die **eine Funktion `classify(...)`**, die wir gleich benutzen.

In [ ]:
# Minimale, eigenständige classify-Funktion für Sitzung 2 (Mock + echt).
# Bewusst einfacher als die volle Pipeline - S2 braucht nur:
# "eine Bewertung rein, Sentiment+Begründung raus".
import json, re

SENTIMENTS = ["positive", "negative", "neutral", "mixed"]

def _mock_classify(text):
    """Attrappe: plausibel, aber deterministisch. Liest Sarkasmus wörtlich (Lehrzweck)."""
    t = (text or "").lower()
    pos = sum(w in t for w in ["gut","top","super","toll","hervorragend","bequem",
                                "stabil","klasse","liebe","genial","perfekt","great","love"])
    neg = sum(w in t for w in ["schlecht","kaputt","teuer","stürzt","rauscht","billig",
                                "nervt","enttäuscht","langsam","mangel","bad","broken"])
    if pos and neg: s = "mixed"
    elif pos: s = "positive"
    elif neg: s = "negative"
    else: s = "neutral"
    return {"sentiment": s, "reason": f"Mock: {pos} positive-, {neg} negative-Signale erkannt."}

def _real_classify(text, client, model="claude-sonnet-5"):
    prompt = (f'Ordne die folgende Produktbewertung ein. Antworte NUR mit JSON:\n'
              f'{{"sentiment": "positive|negative|neutral|mixed", "reason": "kurze Begruendung"}}\n\n'
              f'Bewertung: """{text}"""')
    resp = client.messages.create(model=model, max_tokens=200,
                                  messages=[{"role":"user","content":prompt}])
    raw = next((b.text for b in resp.content if hasattr(b,"text")), "")
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    return json.loads(m.group(0)) if m else {"sentiment":"neutral","reason":"parse error"}

def classify(text, use_real=False, client=None):
    """Die eine Funktion, die ihr aufruft. Standard = Mock; echt mit client."""
    return _real_classify(text, client) if use_real else _mock_classify(text)

SAMPLE_REVIEWS = [
    "Der Klang ist wirklich hervorragend, aber die App stürzt ständig ab.",
    "Absolut top verarbeitet und bequem. Klare Kaufempfehlung!",
    "Nach zwei Wochen kaputt. Nie wieder.",
    "Super, schon nach drei Tagen kaputt. Echt klasse Qualität.",  # sarcasm
    "Ganz okay, erfüllt seinen Zweck.",
]

## 2. Die erste Bewertung einordnen

Jetzt ihr. Führt die Zelle aus — sie ordnet eine Beispiel-Bewertung ein.

In [ ]:
beispiel = SAMPLE_REVIEWS[0]
print('Bewertung:', beispiel)
print('Ergebnis :', classify(beispiel))

> ✏️ **Eure erste Aufgabe:** Ändert `SAMPLE_REVIEWS[0]` in `[1]`, `[2]`, ... (es gibt 5 Beispiele, Index 0–4) und führt erneut aus. Oder schreibt in die nächste Zelle eine **eigene** Bewertung.

In [ ]:
meine_bewertung = "Die Kopfhörer sind bequem, aber der Akku hält kaum."  # <- hier ändern
classify(meine_bewertung)

## 3. Der Blick hinter die Kulissen: der Prompt

Wo ist die „Intelligenz“? In der **Anweisung**, die wir dem Modell schicken — dem *Prompt*. Kein Zauber, nur Text. Hier ist der Prompt, den `classify` beim echten Aufruf verwenden würde:

In [ ]:
beispiel = SAMPLE_REVIEWS[3]   # die sarkastische Bewertung
prompt = (f'Ordne die folgende Produktbewertung ein. Antworte NUR mit JSON:\n'
          f'{{"sentiment": "positive|negative|neutral|mixed", "reason": "kurze Begruendung"}}\n\n'
          f'Bewertung: """{beispiel}"""')
print(prompt)

> ✏️ **Eure Aufgabe:** Das ist der Hebel. Wenn ihr die *Anweisung* ändert, ändert sich das Verhalten. Formuliert den Prompt in der Zelle oben um — z. B. „antworte nur mit einem Wort“ — und überlegt: was würde sich am Ergebnis ändern? *(Ausprobieren mit echtem Schlüssel in Abschnitt 5.)*

## 4. Zweimal dieselbe Frage — dieselbe Antwort?

Führt die nächste Zelle aus: dieselbe Bewertung, zweimal eingeordnet.

In [ ]:
r = SAMPLE_REVIEWS[3]
print('Lauf 1:', classify(r))
print('Lauf 2:', classify(r))

Beim **Mock** kommt zweimal *dasselbe* heraus — er ist deterministisch.

> 🎓 **Jetzt live (Vortragende:r):** Dasselbe mit dem **echten** LLM — und da kann zweimal *Unterschiedliches* herauskommen. Ein LLM ist **nicht deterministisch**: gleiche Eingabe, evtl. andere Ausgabe. Das ist harmlos bei einer Bewertung — aber denkt daran, wenn wir später *messen* wollen, wie gut das Werkzeug ist.

> 💡 **Kernpunkt:** Ein LLM ist kein Taschenrechner. „Richtig“ ist keine feste Größe — ein Faden, der uns die zweite Semesterhälfte begleitet.

## 5. Der echte Aufruf (mit Schlüssel)

Bisher lief alles über den Mock — **ohne Schlüssel**. Für den echten Anthropic-Aufruf braucht es einen API-Key. **Nie** den Schlüssel in den Code oder das Repository schreiben! In Colab kommt er aus dem **Secrets-Panel** (🔑-Symbol links): Name `ANTHROPIC_API_KEY`, Wert einfügen, „Notebook access“ an.

> ℹ️ Wie ihr an einen Kurs-Schlüssel kommt, klären wir in der Sitzung. Ohne Schlüssel überspringt ihr diesen Abschnitt — der Mock reicht für heute.

In [ ]:
# Nur mit Schlüssel ausführen:
from google.colab import userdata
import anthropic
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

r = SAMPLE_REVIEWS[3]
print('Echt, Lauf 1:', classify(r, use_real=True, client=client))
print('Echt, Lauf 2:', classify(r, use_real=True, client=client))

> 🎓 **Beobachtung:** Vergleicht die beiden echten Läufe — und vergleicht das echte Ergebnis mit dem Mock. Wo liegt das LLM richtiger? Wo überrascht es?

## 6. Geschafft — und Ausblick

Ihr habt ein LLM eine Bewertung einordnen lassen. Das ist der kleinste Baustein unseres Produkts.

**Nächste Woche (21.10):** von *einer* Bewertung zu *hunderten* — und die ersten aggregierten Zahlen.

> 💡 Denkt an den Prompt aus Abschnitt 3: Das ganze Semester drehen wir an genau solchen Stellschrauben.